# PlanMargin — reproducible TensorRT qualification

This notebook downloads PlanMargin's 1,024-scenario model-only release, builds FP32 and FP16 TensorRT engines, measures device-only and pinned-host end-to-end latency plus numerical parity, and compiles the C++17 runner. Use a free **T4 GPU** runtime. No WOMD records are downloaded or redistributed; model quality comes from the separately sealed real-WOMD holdout report.

In [ ]:
import subprocess

gpu = subprocess.run(
    [
        "nvidia-smi",
        "--query-gpu=name,memory.total,driver_version",
        "--format=csv,noheader",
    ],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
print(gpu)

In [ ]:
!git clone --depth 1 https://github.com/ethanvillalovoz/planmargin.git
%cd planmargin
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os

os.environ["PATH"] = f"{os.path.expanduser('~')}/.local/bin:{os.environ['PATH']}"
!uv python install 3.11
!uv sync --frozen --extra nvidia
!uv pip install --python .venv/bin/python tensorrt-cu12==11.2.1.2
# PyTorch 2.13's ONNX exporter can import an incompatible CUDA Triton wheel.
# PlanMargin uses eager inference here, so removing Triton avoids that upstream crash.
!uv pip uninstall --python .venv/bin/python triton

## Fetch and verify the deployable model

The model-only release is public and hash-pinned. It contains weights, ONNX, and aggregate training metrics—never source scenarios or per-record outputs.

In [ ]:
!mkdir -p artifacts/experiment-v7/torch-trajectory-model
!curl -L --fail --output artifacts/experiment-v7/torch-trajectory-model/trajectory-model.pmtorch https://github.com/ethanvillalovoz/planmargin/releases/download/trajectory-model-v2/trajectory-model.pmtorch
!curl -L --fail --output artifacts/experiment-v7/torch-trajectory-model/trajectory-model.onnx https://github.com/ethanvillalovoz/planmargin/releases/download/trajectory-model-v2/trajectory-model.onnx
!curl -L --fail --output artifacts/experiment-v7/torch-trajectory-model/training-report.json https://github.com/ethanvillalovoz/planmargin/releases/download/trajectory-model-v2/training-report.json
!printf '%s  %s\n%s  %s\n%s  %s\n' '6e557177c57126fc51fb066033147f316d253126aecb165b3f767d4f04ef8660' 'artifacts/experiment-v7/torch-trajectory-model/trajectory-model.pmtorch' '38934ad17b8ea04698feccf116c6c75a030788124b096c25d99a942386aa73d7' 'artifacts/experiment-v7/torch-trajectory-model/trajectory-model.onnx' '29221c77688cd4dc8078b1f18f68bcd0694e13fe00f8d2f07d9328d8131e0f04' 'artifacts/experiment-v7/torch-trajectory-model/training-report.json' | sha256sum --check

In [ ]:
!.venv/bin/planmargin-qualify-tensorrt --model-dir artifacts/experiment-v7/torch-trajectory-model --output artifacts/experiment-v7/tensorrt-qualification --warmup 50 --iterations 500 --batches 1 8 256
!jq '{status,gates,gpu,environment,measurement,engines}' artifacts/experiment-v7/tensorrt-qualification/qualification-report.json

## Compile and cross-check the C++17 runtime

TensorRT's pip package supplies runtime libraries but not C++ headers. The next cell pins NVIDIA's matching `v11.2` headers, builds the checked-in runner, and measures the same engine.

In [ ]:
import pathlib
import shutil

root = pathlib.Path("/tmp/planmargin-tensorrt")
shutil.rmtree(root, ignore_errors=True)
(root / "lib").mkdir(parents=True)
!git clone --branch v11.2 --depth 1 https://github.com/NVIDIA/TensorRT.git /tmp/TensorRT
shutil.copytree("/tmp/TensorRT/include", root / "include")
venv_site = pathlib.Path(".venv/lib/python3.11/site-packages").resolve()
for library in (venv_site / "tensorrt_libs").glob("libnvinfer.so*"):
    (root / "lib" / library.name).symlink_to(library)
print(root)

In [ ]:
!cmake -S cpp/tensorrt -B build/tensorrt -DTENSORRT_ROOT=/tmp/planmargin-tensorrt -DCMAKE_BUILD_TYPE=Release
!cmake --build build/tensorrt --parallel
!LD_LIBRARY_PATH=/tmp/planmargin-tensorrt/lib:$LD_LIBRARY_PATH build/tensorrt/planmargin_tensorrt_runner --engine artifacts/experiment-v7/tensorrt-qualification/trajectory-fp32.engine --batch 1 --warmup 50 --iterations 500 --mode both | tee artifacts/experiment-v7/tensorrt-qualification/cpp-fp32-batch1.json

In [ ]:
!uv pip freeze --python .venv/bin/python > artifacts/experiment-v7/tensorrt-qualification/environment.lock.txt
!sha256sum artifacts/experiment-v7/torch-trajectory-model/trajectory-model.onnx artifacts/experiment-v7/tensorrt-qualification/trajectory-*.engine > artifacts/experiment-v7/tensorrt-qualification/artifact-sha256.txt
!zip -j artifacts/experiment-v7/planmargin-tensorrt-aggregate.zip artifacts/experiment-v7/tensorrt-qualification/qualification-report.json artifacts/experiment-v7/tensorrt-qualification/cpp-fp32-batch1.json artifacts/experiment-v7/tensorrt-qualification/environment.lock.txt artifacts/experiment-v7/tensorrt-qualification/artifact-sha256.txt
from google.colab import files

files.download("artifacts/experiment-v7/planmargin-tensorrt-aggregate.zip")